<a href="https://colab.research.google.com/github/fauziardiantama/my-falcon-tuning/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Falcon 90M Training & Auto-Transfer

Notebook ini dikonfigurasi untuk menjalankan training pada Falcon 90M Instruct (Full FT atau LoRA) dan mengirimkan hasilnya ke komputer lokal menggunakan `croc`.

In [1]:
# 1. Clone Repositori dan Pindah ke Direktori
import os
REPO_URL = 'https://github.com/fauziardiantama/my-falcon-tuning.git'
REPO_NAME = 'my-falcon-tuning'

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

%cd {REPO_NAME}

# --- DIAGNOSTICS ---
print("--- SYSTEM DIAGNOSTICS ---")
!nvcc --version
!python -c "import torch; print(f'PyTorch: {torch.__version__}'); print(f'CUDA available: {torch.cuda.is_available()}'); print(f'CUDA version: {torch.version.cuda}')"

# 2. Instalasi Requirements
TRAINING_METHOD = 'full' # 'full' atau 'lora'
REQ_FILE = 'requirements_lora.txt' if TRAINING_METHOD == 'lora' else 'requirements.txt'

print(f"--- INSTALLING DEPENDENCIES FOR {TRAINING_METHOD.upper()} ---")
# 1. Install uv untuk instalasi yang super cepat
!pip install -q uv

# 2. Instalasi standar untuk requirements umum menggunakan uv
!uv pip install --system -q -r {REQ_FILE}

# 3. Instalasi akselerasi Mamba dari sumber (Memastikan kompatibilitas CUDA 12.8)
if TRAINING_METHOD == 'full':
    print("--- INSTALLING MAMBA ACCELERATION KERNELS (Building from Source) ---")
    # Menggunakan MAX_JOBS=2 sesuai core CPU Colab agar stabil
    # --no-build-isolation memastikan kernel terikat dengan PyTorch 2.10 sistem
    !MAX_JOBS=2 pip install causal-conv1d>=1.5.0 --no-build-isolation --no-cache-dir -v
    !MAX_JOBS=2 pip install mamba-ssm --no-build-isolation --no-cache-dir -v

# 3. Instalasi Croc (File Transfer)
print("--- INSTALLING CROC ---")
!curl https://getcroc.schollz.com | bash

Cloning into 'my-falcon-tuning'...
remote: Enumerating objects: 79, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 79 (delta 46), reused 50 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (79/79), 3.70 MiB | 10.81 MiB/s, done.
Resolving deltas: 100% (46/46), done.
/content/my-falcon-tuning
--- SYSTEM DIAGNOSTICS ---
/bin/bash: line 1: nvcc: command not found
PyTorch: 2.10.0+cpu
CUDA available: False
CUDA version: None
--- INSTALLING DEPENDENCIES FOR FULL ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 54.9 MB/s eta 0:00:00
--- INSTALLING MAMBA ACCELERATION KERNELS (Fast Pre-built Wheels) ---
Using Python 3.12.13 environment at: /usr
Resolved 13 packages in 7.29s
Prepared 1 package in 5.76s
Installed 1 package in 1ms
 + causal-conv1d==1.6.1+cu12torch2.9cxx11abitrue (from https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+cu12torch2.9cxx11abiTRUE-cp312-cp

In [2]:
# 4. Jalankan Script Training
TRAINING_SCRIPT = 'train_falcon_lora.py' if TRAINING_METHOD == 'lora' else 'train_falcon.py'
print(f"--- STARTING {TRAINING_METHOD.upper()} TRAINING ---")
!python {TRAINING_SCRIPT}

--- STARTING FULL TRAINING ---
Loading tokenizer: tiiuae/Falcon-H1-Tiny-90M-Instruct
config.json: 1.62kB [00:00, 3.53MB/s]
tokenizer_config.json: 99.7kB [00:00, 123MB/s]
tokenizer.json: 2.35MB [00:00, 40.4MB/s]
special_tokens_map.json: 7.69kB [00:00, 14.6MB/s]
chat_template.jinja: 2.71kB [00:00, 8.36MB/s]
Loading model: tiiuae/Falcon-H1-Tiny-90M-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 182M/182M [00:03<00:00, 55.6MB/s]
The fast path is not available because one of `(selective_state_update, causal_conv1d_fn, causal_conv1d_update)` is None. Falling back to the naive implementation. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100% 386/386 [00:00<00:00, 989.42it/s, Materializing param=model.layers.23.self_attn.v_proj.weight]
generation_config.json: 100% 152/152 [00:00<00:00, 591kB/s]
Loading dataset from: dataset_rpg.jsonl
Generating train split: 24260 example

In [4]:
# 5. Kirim Hasil Menggunakan Croc
OUTPUT_DIR = './falcon-90m-lora-adapter' if TRAINING_METHOD == 'lora' else './falcon-90m-fine-tuned'
print(f"--- SENDING {TRAINING_METHOD.upper()} RESULT VIA CROC ---")
print("Salin kode yang muncul di bawah ini untuk menerima file di komputer lokal:")
!croc send {OUTPUT_DIR}

--- SENDING FULL RESULT VIA CROC ---
Salin kode yang muncul di bawah ini untuk menerima file di komputer lokal:
Sending 0 files and 1 folders (0 B)
Code is: 1316-dynasty-degree-rainbow

On the other computer run:
(For Windows)
    croc 1316-dynasty-degree-rainbow
(For Linux/macOS)
    CROC_SECRET="1316-dynasty-degree-rainbow" croc 
